## 1. So sánh Tổng quan

### 1.1. So sánh Kiến trúc

| Tiêu chí | CNN Tự Xây Dựng | ResNet-50 (Transfer Learning) | Nhận xét |
|----------|-----------------|-------------------------------|----------|
| **Kiến trúc** | 3-4 Conv layers + Pooling + Dense | 50 layers với Residual Connections | ResNet sâu hơn, phức tạp hơn |
| **Số Parameters** | ~2-5 triệu | ~25 triệu (trainable: 23M+) | ResNet có khả năng học đặc trưng phức tạp hơn |
| **Pretrained Weights** | Train from scratch | ImageNet pretrained | Lợi thế lớn từ kiến thức đã học |
| **Thời gian Train** | Nhanh hơn (~10-15 phút) | Chậm hơn (~30-45 phút) | Trade-off giữa tốc độ và độ chính xác |
| **Test Accuracy** | ~85-90% (ước tính) | ~92-95% | ResNet vượt trội nhờ Transfer Learning |
| **Recall (Sensitivity)** | Thấp hơn | Cao hơn (~95-98%) | Quan trọng trong y tế - giảm bỏ sót bệnh |
| **Overfitting Risk** | Cao hơn (ít data) | Thấp hơn (pretrained features) | ResNet tổng quát hóa tốt hơn |
| **Khả năng Giải thích** | Khó phân tích | Grad-CAM rõ ràng | ResNet cho phép visualize attention maps |

### 1.2. Import Libraries và Load Metrics

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Thiết lập style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

### 1.3. Load Kết quả từ Các Mô hình

**Lưu ý:** Cần chạy `CNN.ipynb` và `ResNet.ipynb` trước để tạo file kết quả trong `./reports/`

In [ ]:
# Load kết quả CNN (nếu có)
try:
    with open('./reports/cnn_metrics.json', 'r') as f:
        cnn_metrics = json.load(f)
    print("Đã load CNN metrics")
except FileNotFoundError:
    print("Chưa có file ./reports/cnn_metrics.json")
    print("   Vui lòng chạy CNN.ipynb và export metrics trước.")
    cnn_metrics = None

# Load kết quả ResNet
try:
    with open('./reports/resnet50_metrics.json', 'r') as f:
        resnet_metrics = json.load(f)
    print("Đã load ResNet-50 metrics")
except FileNotFoundError:
    print("Chưa có file ./reports/resnet50_metrics.json")
    print("   Vui lòng chạy ResNet.ipynb và export metrics trước.")
    resnet_metrics = None

## 2. So sánh Hiệu suất Chi tiết

### 2.1. Bảng So sánh Metrics

In [ ]:
if cnn_metrics and resnet_metrics:
    # Tạo DataFrame so sánh
    comparison_data = {
        'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'MCC', 'Cohen Kappa', 'G-Mean'],
        'CNN': [
            cnn_metrics.get('Accuracy', 'N/A'),
            cnn_metrics.get('Precision', 'N/A'),
            cnn_metrics.get('Recall', 'N/A'),
            cnn_metrics.get('F1-Score', 'N/A'),
            cnn_metrics.get('ROC-AUC', 'N/A'),
            cnn_metrics.get('MCC', 'N/A'),
            cnn_metrics.get('Cohen Kappa', 'N/A'),
            cnn_metrics.get('G-Mean', 'N/A')
        ],
        'ResNet-50': [
            resnet_metrics.get('Accuracy', 'N/A'),
            resnet_metrics.get('Precision', 'N/A'),
            resnet_metrics.get('Recall', 'N/A'),
            resnet_metrics.get('F1-Score', 'N/A'),
            resnet_metrics.get('ROC-AUC', 'N/A'),
            resnet_metrics.get('MCC', 'N/A'),
            resnet_metrics.get('Cohen Kappa', 'N/A'),
            resnet_metrics.get('G-Mean', 'N/A')
        ]
    }
    
    df_comparison = pd.DataFrame(comparison_data)
    
    # Tính chênh lệch
    def calc_diff(cnn_val, resnet_val):
        if cnn_val == 'N/A' or resnet_val == 'N/A':
            return 'N/A'
        diff = resnet_val - cnn_val
        return f"{diff:+.4f}"
    
    df_comparison['Diff (ResNet - CNN)'] = df_comparison.apply(
        lambda row: calc_diff(row['CNN'], row['ResNet-50']), axis=1
    )
    
    print("\nSo sánh Metrics giữa CNN và ResNet-50:\n")
    print(df_comparison.to_string(index=False))
else:
    print("Thiếu dữ liệu metrics. Vui lòng chạy cả CNN.ipynb và ResNet.ipynb trước.")

### 2.2. Biểu đồ So sánh Metrics

In [ ]:
if cnn_metrics and resnet_metrics:
    # Chọn các metrics chính để visualize
    metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
    cnn_values = [cnn_metrics.get(m, 0) for m in metrics_to_plot]
    resnet_values = [resnet_metrics.get(m, 0) for m in metrics_to_plot]
    
    # Vẽ biểu đồ
    x = np.arange(len(metrics_to_plot))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(12, 6))
    bars1 = ax.bar(x - width/2, cnn_values, width, label='CNN', alpha=0.8)
    bars2 = ax.bar(x + width/2, resnet_values, width, label='ResNet-50', alpha=0.8)
    
    ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
    ax.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax.set_title('So sánh Hiệu suất: CNN vs ResNet-50', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_to_plot)
    ax.legend()
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    
    # Thêm giá trị lên trên mỗi cột
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}',
                   ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('./reports/model_comparison.png', dpi=300, bbox_inches='tight')
    print("Đã lưu biểu đồ tại ./reports/model_comparison.png")
    plt.show()
else:
    print("Không thể vẽ biểu đồ do thiếu dữ liệu.")

## 3. Phân tích Ưu và Nhược điểm

### 3.1. Ưu điểm của ResNet-50

1. **Transfer Learning mạnh mẽ**: Tận dụng kiến thức từ ImageNet (1.4M ảnh) giúp mô hình nhận diện đặc trưng tốt hơn ngay cả với dataset nhỏ.
2. **Residual Connections**: Giải quyết vấn đề gradient vanishing, cho phép huấn luyện mạng sâu hiệu quả.
3. **Độ chính xác cao**: Đặc biệt ở chỉ số Recall (quan trọng trong y tế để giảm False Negative).
4. **Khả năng tổng quát hóa**: Ít bị overfitting hơn nhờ pretrained weights và data augmentation.

### 3.2. Nhược điểm của ResNet-50

1. **Tài nguyên lớn**: Yêu cầu GPU mạnh, bộ nhớ lớn (~4-8GB VRAM).
2. **Thời gian train dài**: Mặc dù chỉ fine-tune nhưng vẫn chậm hơn CNN nhỏ.
3. **Model size lớn**: File .pth ~98MB, khó deploy trên thiết bị edge.
4. **Black box**: Mặc dù có Grad-CAM nhưng vẫn khó giải thích hoàn toàn quyết định của mô hình.

### 3.3. Ưu điểm của CNN Tự Xây Dựng

1. **Nhẹ và nhanh**: Train nhanh, inference nhanh, phù hợp với tài nguyên hạn chế.
2. **Model size nhỏ**: Dễ deploy trên edge devices (Raspberry Pi, mobile).
3. **Kiểm soát được**: Dễ điều chỉnh kiến trúc theo nhu cầu cụ thể.
4. **Ít tài nguyên**: Có thể train trên CPU hoặc GPU nhỏ.

### 3.4. Nhược điểm của CNN Tự Xây Dựng

1. **Độ chính xác thấp hơn**: Đặc biệt với dataset nhỏ.
2. **Overfitting risk cao**: Cần regularization mạnh.
3. **Feature extraction hạn chế**: Không có kiến thức pretrained.
4. **Cần dataset lớn**: Để đạt hiệu suất tương đương ResNet.

## 4. Kết luận và Đề xuất

### 4.1. Khi nào chọn ResNet-50?

**ResNet-50 với Transfer Learning là lựa chọn tối ưu** khi:
- Yêu cầu độ chính xác cao (đặc biệt Recall)
- Có sẵn GPU để train
- Dataset không quá lớn (Transfer Learning hiệu quả với ít data)
- Ưu tiên hiệu suất hơn tốc độ inference
- Ứng dụng trong môi trường y tế (hospital server)

### 4.2. Khi nào chọn CNN Tự Xây Dựng?

**CNN tự xây dựng phù hợp hơn** khi:
- Tài nguyên hạn chế (CPU only)
- Cần deploy trên thiết bị edge/mobile
- Yêu cầu inference nhanh (real-time screening)
- Dataset đủ lớn để train from scratch
- Chi phí cloud/GPU là vấn đề

### 4.3. Đề xuất Hybrid Approach

Có thể kết hợp cả hai:
1. **ResNet-50** cho chẩn đoán chính xác (bác sĩ review)
2. **CNN nhỏ** cho screening ban đầu (edge device)
3. **Ensemble** cả hai để tăng độ tin cậy

### 4.4. Next Steps

- [ ] Thử thêm các mô hình khác: EfficientNet, DenseNet, Vision Transformer
- [ ] Ensemble methods để tăng accuracy
- [ ] Model compression (Quantization, Pruning) cho ResNet
- [ ] External validation trên dataset khác

---

## Tài liệu Tham khảo

1. He, K., et al. (2016). "Deep Residual Learning for Image Recognition"
2. Rajpurkar, P., et al. (2017). "CheXNet: Radiologist-Level Pneumonia Detection"
3. Transfer Learning in Medical Imaging: A Survey (2020)